# RegWatch: decision predictor (BiomedBERT / PubMedBERT)
Runtime → Change runtime type → **T4 GPU**. Run `python predict.py` locally first, then upload `data/predictor_split.csv` when asked. Same split and metrics as the TF-IDF baseline in `results/predictor_eval.csv`.

In [ ]:
!pip -q install transformers datasets accelerate
from google.colab import files
files.upload()  # pick predictor_split.csv

In [ ]:
import numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score, cohen_kappa_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract'  # formerly PubMedBERT
df = pd.read_csv('predictor_split.csv')
labels = sorted(df.label.unique()); lid = {l: i for i, l in enumerate(labels)}
df['y'] = df.label.map(lid)
tok = AutoTokenizer.from_pretrained(MODEL)
enc = lambda d: Dataset.from_pandas(d[['request_text', 'y']].rename(columns={'y': 'labels'}).reset_index(drop=True)).map(
    lambda b: tok(b['request_text'], truncation=True, max_length=256), batched=True)
train, test, gold = (enc(df[df.split == s]) for s in ['train', 'test', 'gold'])
print({s: (df.split == s).sum() for s in ['train', 'test', 'gold']})

In [ ]:
# class weights: decisions are imbalanced, same idea as class_weight='balanced' in the baseline
w = torch.tensor(len(df[df.split == 'train']) / (len(labels) * np.bincount(df[df.split == 'train'].y, minlength=len(labels))), dtype=torch.float)
class WTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        y = inputs.pop('labels'); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, y, weight=w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels))
args = TrainingArguments('out', num_train_epochs=3, per_device_train_batch_size=16, learning_rate=2e-5,
                         weight_decay=0.01, fp16=True, save_strategy='no', report_to='none', seed=0)
trainer = WTrainer(model=model, args=args, train_dataset=train, tokenizer=tok)
trainer.train()

In [ ]:
for name, ds in [('test', test), ('gold', gold)]:
    if len(ds) == 0: continue
    p = trainer.predict(ds).predictions.argmax(-1); y = np.array(ds['labels'])
    print({'model': f'biomedbert ({name})', 'n': len(y), 'macro_f1': round(f1_score(y, p, average='macro'), 3),
           'kappa': round(cohen_kappa_score(y, p), 3)})
# paste these two lines back to Claude for results/predictor_eval.csv